# Cross-domain engineering SVG patch training: 7B on A100

Trains Qwen2.5-Coder-7B-Instruct LoRA on 8,000 lineage-disjoint SVG edits and evaluates 100 held-out examples. Select an A100 GPU runtime and run all cells. Results are written to Colab local storage; download the artifacts before the runtime expires.


In [ ]:
import subprocess, sys, torch

def run(args, **kw):
    print('RUN:', ' '.join(map(str, args)), flush=True)
    p = subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                         bufsize=1, **kw)
    for line in p.stdout:
        print(line, end='', flush=True)
    if p.wait():
        raise RuntimeError('command failed: ' + ' '.join(map(str, args)))

assert torch.cuda.is_available(), 'Connect a GPU runtime first.'
assert 'A100' in torch.cuda.get_device_name(0), 'This run requires an A100 runtime.'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# Fetch from GitHub rather than uploading: files.upload() is a browser-only widget that the VS Code
# notebook renderer disables, which is what stalled the previous attempt.
import pathlib
R = '/content/SVG'
if pathlib.Path(R, '.git').exists():
    run(['git', '-C', R, 'fetch', '-q', '--depth', '1', 'origin', 'cvpr2027-research-package'])
    run(['git', '-C', R, 'reset', '-q', '--hard', 'FETCH_HEAD'])
else:
    run(['git', 'clone', '-q', '--depth', '1', '-b', 'cvpr2027-research-package',
         '--filter=blob:none', '--sparse', 'https://github.com/ayushdebnath012/SVG.git', R])
run(['git', '-C', R, 'sparse-checkout', 'set', 'cvpr2027/scripts', 'cvpr2027/src',
     'cvpr2027/data/engsvg-crossdomain-edit-v1'])
run(['git', '-C', R, 'log', '--oneline', '-1'])

In [ ]:
run([sys.executable, '-m', 'pip', 'install', '-q', 'transformers>=4.48', 'peft>=0.14',
     'accelerate>=1.2', 'svgpathtools==1.7.2'])
# Colab preinstalls torchao 0.10 and peft's LoRA dispatch raises on anything below 0.16.
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', '-q', 'torchao'], check=False)

In [ ]:
from pathlib import Path
Path('/content/engsvg-crossdomain-7b-a100-run').mkdir(parents=True, exist_ok=True)


In [ ]:
import os
project = '/content/SVG/cvpr2027'
env = dict(os.environ, PYTHONPATH='src:scripts')
run([sys.executable, 'scripts/train_crossdomain_svg_patcher.py',
     '--data', 'data/engsvg-crossdomain-edit-v1',
     '--out', '/content/engsvg-crossdomain-7b-a100-run',
     '--model', 'Qwen/Qwen2.5-Coder-7B-Instruct', '--resume',
     '--epochs', '1', '--eval-count', '100',
     '--max-length', '4096', '--max-new-tokens', '1400'], cwd=project, env=env)

In [ ]:
import json, pathlib
from google.colab import files
s = json.loads(pathlib.Path('/content/engsvg-crossdomain-7b-a100-run/summary.json').read_text())
print(json.dumps(s, indent=2)[:4000])
files.download('/content/engsvg-crossdomain-7b-a100-run-artifacts.zip')
